
# Exercises — Agent Architectures

These activities reinforce the core ideas presented in the Agent Architectures chapter: the PEAS framework, the distinction between the agent function and the agent program, why table-driven agents are impractical, the progression from reflex to model-based, goal-based, and utility-based agents, rationality as expected utility maximization, and the boid model as a concrete example of a state-update architecture.

## Exercise 1 — PEAS and environment classification

**Problem**

Consider an autonomous delivery robot that drives around a warehouse, picks up parcels from shelves, and drops them at packing stations, sharing the floor with human workers and other robots. (A) Give a PEAS description (Performance measure, Environment, Actuators, Sensors), and (B) classify the task environment along the six axes: observable, single/multi-agent, deterministic/stochastic, static/dynamic, discrete/continuous, known/unknown, justifying each choice in one line.

**Solution.**

**Step 1 — PEAS.** We choose measurable quantities aligned with what we *actually want* (per the chapter's warning against rewarding the wrong thing).

| Component | Specification |
|---|---|
| **Performance** | parcels delivered correctly per hour; time/energy per delivery; number of collisions or near-misses (penalised) |
| **Environment** | warehouse floor, shelves, parcels, packing stations, human workers, other robots |
| **Actuators** | drive motors (wheels), steering, gripper/lift arm, signalling lights/speaker |
| **Sensors** | cameras, LIDAR/range finders, wheel odometry, gripper force/contact sensors, battery gauge |

**Step 2 — Environment classification.**

| Axis | Answer | Justification |
|---|---|---|
| Observable | **Partially** | on-board sensors see only the local surroundings, not the whole warehouse |
| Agents | **Multi-agent** | humans and other robots act and their behaviour affects the robot |
| Deterministic | **Stochastic** | wheel slip, sensor noise, and unpredictable humans make outcomes uncertain |
| Static / Dynamic | **Dynamic** | the world changes while the robot deliberates (people move) |
| Discrete / Continuous | **Continuous** | positions, velocities and time vary smoothly |
| Known / Unknown | **Partially known** | physics of driving is known; the live positions of people/parcels must be learned online |

> **Key concept.** PEAS forces us to make the *performance measure* explicit and measurable **before** designing the agent, and the six axes tell us *how hard* the environment is — here the hardest combination (partially observable, multi-agent, stochastic, dynamic, continuous), exactly like autonomous driving.


## Exercise 1.2 — Size of the lookup table

**Problem.** A *table-driven agent* stores one action for every possible **percept sequence**. Let $P = |\mathcal{P}|$ be the number of distinct percepts and let the agent live for $T$ time steps (so percept sequences have length $1$ to $T$).

1. Derive a closed-form expression for the number of table entries $E$.
2. Evaluate it for the vacuum world, where a percept is a (location, status) pair with $\mathcal{P} = \{A,B\}\times\{clean,dirty\}$, so $P=4$, and $T=3$.
3. Comment on why this makes table-driven design impossible in general.


**Solution.**

**Step 1 — Count the sequences.** There are $P^{t}$ distinct percept sequences of length exactly $t$ (each of the $t$ positions can be any of $P$ percepts). Summing over all lengths $t = 1,\dots,T$:

$$E = \sum_{t=1}^{T} P^{t}.$$

This is a finite geometric series with ratio $P$. Using $\sum_{t=1}^{T} x^t = \dfrac{x(x^{T}-1)}{x-1}$:

$$\boxed{\,E = \frac{P\,(P^{T}-1)}{P-1}\,}.$$

**Step 2 — Evaluate.** With $P=4$ and $T=3$:

$$E = 4 + 4^2 + 4^3 = 4 + 16 + 64 = 84,$$

or via the closed form $E = \dfrac{4\,(4^{3}-1)}{4-1} = \dfrac{4\cdot 63}{3} = 84$.

**Step 3 — Interpret.** $E$ grows **exponentially** in the horizon $T$. Even this toy vacuum world already needs 84 entries for only three steps; the chapter's driving example (a single camera, one hour) yields on the order of $10^{6\times10^{11}}$ entries — vastly more than the $\sim 10^{80}$ atoms in the observable universe.

> **Key concept.** The *agent function* (percept-sequence → action) is a well-defined mathematical object, but representing it explicitly as a table is impossible. This is exactly why we need a compact **agent program** (e.g. a reflex rule) that *computes* the same mapping without storing it.


## Exercise 1.3 — Why a reflex agent may need randomisation

**Problem.** A vacuum agent perceives only *(current location status)* — it cannot tell whether it is in square $A$ or $B$. Both squares are clean. A **deterministic** reflex agent must map the percept `clean` to a single action.

1. Show that any deterministic reflex agent can fail (loop forever) in this partially observable world.
2. Now consider a **randomised** reflex agent that, on `clean`, moves *left* or *right* each with probability $\tfrac12$. Model the number of steps needed to reach the other square and compute its **expected value**.


**Solution.**

**Step 1 — Deterministic failure.** With only the percept `clean`, a deterministic rule fixes one action, say always `right`. If the agent starts in the rightmost square, `right` bumps against the wall and it stays put; it perceives `clean` again and repeats forever — an **infinite loop**. Symmetrically for `left`. No deterministic reflex rule works from *both* squares because the agent cannot distinguish them.

**Step 2 — Randomised agent.** At each step the agent moves toward the other square with probability $p=\tfrac12$ (a successful "cross") and stays otherwise (bumping the wall or moving away then back — model it as a Bernoulli success per step). Let $N$ be the number of steps until the first success. Then $N$ is **geometric** with parameter $p$:

$$\Pr\{N = k\} = (1-p)^{\,k-1} p, \qquad k = 1, 2, 3, \dots$$

Its expectation is

$$\mathbb{E}[N] = \sum_{k=1}^{\infty} k\,(1-p)^{k-1} p = \frac{1}{p}.$$

*Derivation of $\mathbb{E}[N]=1/p$:* condition on the first step. With prob. $p$ we succeed in $1$ step; with prob. $1-p$ we have "wasted" one step and face the same problem again:

$$\mathbb{E}[N] = p\cdot 1 + (1-p)\,(1 + \mathbb{E}[N]) \;\Rightarrow\; \mathbb{E}[N] = 1 + (1-p)\,\mathbb{E}[N] \;\Rightarrow\; \mathbb{E}[N] = \tfrac{1}{p}.$$

With $p=\tfrac12$: $\ \mathbb{E}[N] = 2$ steps on average.

> **Key concept.** Under **partial observability** a deterministic reflex agent can be strictly worse than a randomised one. Randomisation breaks symmetric traps — a first glimpse of why *stochastic* behaviour (and, later, *exploration*) is rational.


## Exercise 1.4 — Rational action = maximising *expected* utility

**Problem.** A utility-based agent must choose between two actions in an uncertain environment. The designer's utility function $U$ assigns:

- **Action $a_1$ (risky):** with probability $0.8$ the outcome has utility $U=+10$; with probability $0.2$ it has utility $U=-5$.
- **Action $a_2$ (safe):** deterministically yields utility $U=+6$.

1. Compute the **expected utility** of each action and state the rational choice.
2. An *omniscient* agent knows in advance that this particular time the risky action would give $-5$. What would it do, and why does that not make $a_1$ irrational?


**Solution.**

**Step 1 — Expected utilities.** The expected utility of an action is $\mathbb{E}[U] = \sum_{\text{outcomes}} \Pr(\text{outcome})\,U(\text{outcome})$:

$$\mathbb{E}[U\mid a_1] = 0.8\times 10 + 0.2\times(-5) = 8 - 1 = 7.0,$$
$$\mathbb{E}[U\mid a_2] = 1.0\times 6 = 6.0.$$

Since $\mathbb{E}[U\mid a_1] = 7.0 > 6.0 = \mathbb{E}[U\mid a_2]$, the **rational choice is $a_1$**.

**Step 2 — Rationality vs. omniscience.** An *omniscient* agent, knowing the hidden outcome will be $-5$, would pick $a_2$ and score $+6$ instead of $-5$. But omniscience is impossible: rationality is judged on the *information available at decision time*. Given only the probabilities, $a_1$ has the higher **expected** utility and is therefore the rational decision, even though on this particular occasion it turned out worse.

> **Key concept.** A rational (utility-based) agent maximises **expected** utility, not actual utility. Rationality ≠ perfection; the gap is exactly the unavoidable uncertainty of the environment.


## Exercise 1.5 — One step of a boid (Euler integration)

**Problem.** A boid (model-based agent) has mass $m=1$ and, at time $t$, position $\mathbf{p}(t)=(0,0)$ and velocity $\mathbf{v}(t)=(1,0)$. Its three steering rules produce the (unit) force vectors

$$\mathbf{F}_{\text{sep}}=(1,0),\quad \mathbf{F}_{\text{coh}}=(0,-1),\quad \mathbf{F}_{\text{align}}=(1,1),$$

combined with weights $w_{\text{sep}}=1.2,\ w_{\text{coh}}=0.5,\ w_{\text{align}}=1.0$ (no external force). Using the explicit Euler rule from the chapter with $\Delta t = 1$,

$$\mathbf{v}(t+\Delta t)=\mathbf{v}(t)+\mathbf{a}(t)\,\Delta t, \qquad \mathbf{p}(t+\Delta t)=\mathbf{p}(t)+\mathbf{v}(t)\,\Delta t, \qquad \mathbf{a}=\mathbf{F}/m,$$

compute the boid's state after **one** update (and, as a check, after a second update).


**Solution.**

**Step 1 — Total force.** Weighted sum of the steering forces:

$$\mathbf{F} = 1.2\,(1,0) + 0.5\,(0,-1) + 1.0\,(1,1) = (1.2+1.0,\ -0.5+1.0) = (2.2, 0.5).$$

**Step 2 — Acceleration.** With $m=1$: $\ \mathbf{a} = \mathbf{F}/m = (2.2, 0.5)$.

**Step 3 — Euler update (first step).**

$$\mathbf{v}(t{+}1) = (1,0) + (2.2, 0.5)\cdot 1 = (3.2, 0.5),$$
$$\mathbf{p}(t{+}1) = (0,0) + (1,0)\cdot 1 = (1, 0).$$

(Note the position uses the **old** velocity $(1,0)$, as written in the chapter's equations.)

**Step 4 — Second step (check).** Now $\mathbf{v}=(3.2, 0.5)$, $\mathbf{p}=(1, 0)$, same acceleration $\mathbf{a}=(2.2, 0.5)$:

$$\mathbf{v}(t{+}2) = (3.2, 0.5) + (2.2, 0.5) = (5.4, 1), \qquad \mathbf{p}(t{+}2) = (1, 0) + (3.2, 0.5) = (4.2, 0.5).$$

> **Key concept.** A model-based agent *is* its state-update rule: perceive neighbours → compute forces → integrate to a new state. Complex flocking emerges from repeatedly applying this simple local update — no global plan required. (Aside: the chapter's *code* uses the semi-implicit variant, updating position with the **new** velocity; be consistent about which one you use.)
